## REED 

### Setup Background

In [1]:
import sys
sys.path.append(r'c:\Users\TimWeber\repos_20LCA\REED\reed\src')

In [2]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config

import matplotlib.pyplot as plt
import d3graph
import polyviz

11:55:09+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [3]:
config = Config()

In [4]:
bd.projects.set_current(config.PROJECT_NAME)

In [5]:
import import_db

In [6]:
bd.databases

Databases dictionary with 4 object(s):
	bonsai_231231
	bonsai_231231 biosphere
	ecoinvent-3.10-biosphere
	reed

### Install and Explore BONSAI

In [22]:
from bw2io.importers import ExcelImporter
from reed.utils import Config

importer  = ExcelImporter(config.REED_PATH/'Data'/'Data_Collection'/'Data_collection_EUREED_20250710_tw.xlsx',)
importer.match_database(fields=['name','unit','location'])
importer.match_database(db_name=bonsai.name,fields=['name','unit','location','activity type'])
importer.write_database()

Extracted 2 worksheets in 0.08 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
12:26:19+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 7/7 [00:00<00:00, 973.90it/s]


12:26:19+0200 [info     ] Vacuuming database            
Created database: reed


In [23]:
bonsai = bd.Database('bonsai_231231')
bonsai_bio = bd.Database('bonsai_231231 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')
reed = bd.Database('reed')

In [24]:
pd.DataFrame(importer.unlinked)

""


In [25]:
data_reed = [activity.as_dict() for activity in reed]
df_reed = pd.DataFrame(data_reed)


In [26]:
from IPython.display import display
display(df_reed)

,code,comment,location,production amount,reference product,unit,name,worksheet name,database,type,id
0,steel production,proxy,CN,1,steel production,kilogram,steel production,LCA,reed,processwithreferenceproduct,38933
1,cast iron production,proxy,CN,1,cast iron production,kilogram,cast iron production,LCA,reed,processwithreferenceproduct,38934
2,electricity production,proxy,ES,1,electricity production,kilowatt hour,electricity production,LCA,reed,processwithreferenceproduct,38935
3,"soraluce machine, Komatsu",This process represents the materials for the ...,GB,1,"soraluce machine, Komatsu",unit,"soraluce machine, Komatsu",LCA,reed,processwithreferenceproduct,38936
4,"soraluce machine, Goimek",This process represents the materials for the ...,DE,1,"soraluce machine, Goimek",unit,"soraluce machine, Goimek",LCA,reed,processwithreferenceproduct,38938
5,"soraluce machine, WWR",This process represents the materials for the ...,DE,1,"soraluce machine, WWR",unit,"soraluce machine, WWR",LCA,reed,processwithreferenceproduct,38937
6,welding,process modelled as foreground activity,Global,1,welding,meter,welding,LCA,reed,processwithreferenceproduct,38932


### LCA scores

In [15]:
bonsai.search('plastic de')

['rubber and plastic products' (tonne, DE, None),
 'rubber and plastic products' (tonne, DE, None),
 'treatment of waste/scrap, plastic' (tonne, DE, None),
 'recycling of plastic and rubber' (tonne, DE, None)]

In [27]:
reed.search('steel')

['steel production' (kilogram, CN, None)]

In [28]:
steel = reed.get(name = 'steel production', location = "CN", unit = 'kilogram')
welding = reed.get(name = 'welding', location = "Global", unit = 'meter')
soraluce_wwr = reed.get(name = 'soraluce machine, WWR', location = "DE", unit = 'unit')

In [29]:
steel_lca = steel.lca(config.lcia_gwp100)
steel_lca.score

3.699659111742151

In [30]:
welding_lca = welding.lca(config.lcia_gwp100)
welding_lca.score

0.18542379320115848

In [31]:
soraluce_wwr_lca = soraluce_wwr.lca(config.lcia_gwp100)
soraluce_wwr_lca.score

291049.7307757704

### Contribution analysis

In [32]:
bwa.print_recursive_calculation(steel,
lcia_method=config.lcia_gwp100,max_level=5,cutoff=0.01)

Fraction of score | Absolute score | Amount | Activity
0001 |   3.7 |     1 | 'steel production' (kilogram, CN, None)
  0001 |   3.7 | 0.001 | 'basic iron and steel and of ferro-alloys and first products thereof' (tonne, CN, None)
    -0.995 | -3.681 | -0.0009906 | 'basic iron and steel and of ferro-alloys and first products thereof' (tonne, CN, None)


c:\Users\TimWeber\anaconda3\envs\reed\Lib\site-packages\bw2analyzer\utils.py:154: UserWarning: Hit multiple production exchanges; aborting in this branch
  warn("Hit multiple production exchanges; aborting in this branch")


In [33]:
bwa.print_recursive_calculation(soraluce_wwr,
lcia_method=config.lcia_gwp100,max_level=5,cutoff=0.01)

Fraction of score | Absolute score | Amount | Activity
0001 | 2.91e+05 |     1 | 'soraluce machine, WWR' (unit, DE, None)
  0.29 | 8.454e+04 | 2.285e+04 | 'steel production' (kilogram, CN, None)
    0.29 | 8.454e+04 | 22.85 | 'basic iron and steel and of ferro-alloys and first products thereof' (tonne, CN, None)
      -0.289 | -8.412e+04 | -22.63 | 'basic iron and steel and of ferro-alloys and first products thereof' (tonne, CN, None)
  0.692 | 2.013e+05 | 5.44e+04 | 'cast iron production' (kilogram, CN, None)
    0.692 | 2.013e+05 |  54.4 | 'basic iron and steel and of ferro-alloys and first products thereof' (tonne, CN, None)
      -0.688 | -2.003e+05 | -53.89 | 'basic iron and steel and of ferro-alloys and first products thereof' (tonne, CN, None)
